In [24]:
import pandas as pd
import numpy as np
from datetime import datetime

In [25]:
dataset = pd.read_csv(r'../data/NepalLatestAQI.csv')
dataset.head()

,date,station,latitude,longitude,aqi,pm2_5,pm10,no2,so2,o3,temperature_C,relative_humidity_%,notes
0,2024-01-01,Kathmandu,27.7172,85.3240,162,75.9,161.5,18.5,8.0,16.2,20.9,65.2,NaN
1,2024-01-01,Lalitpur,27.6648,85.3188,157,66.9,105.9,18.2,4.2,34.1,28.8,76.2,NaN
2,2024-01-01,Bhaktapur,27.6714,85.4270,166,84.8,182.9,16.4,7.1,26.7,20.3,60.9,NaN
3,2024-01-01,Pokhara,28.2096,83.9856,79,25.5,47.8,2.0,1.0,31.7,23.0,47.9,NaN
4,2024-01-01,Biratnagar,26.4525,87.2718,146,53.8,86.3,17.2,1.9,34.5,23.9,54.1,NaN


In [26]:
dataset.isnull().sum()

date                      0
station                   0
latitude                  0
longitude                 0
aqi                       0
pm2_5                     0
pm10                      0
no2                       0
so2                       0
o3                        0
temperature_C             0
relative_humidity_%       0
notes                  9761
dtype: int64

In [27]:
dataset = dataset.drop(columns='notes')

In [28]:
dataset = dataset.sort_values('date').drop_duplicates(subset=['station', 'date'], keep='last')

In [29]:
dataset['date'] = pd.to_datetime(dataset['date'])
dataset['month'] = dataset['date'].dt.month
dataset['day'] = dataset['date'].dt.day
dataset['dayofweek'] = dataset['date'].dt.dayofweek
dataset['is_weekend'] = dataset['dayofweek'].isin([5,6]).astype(int)

In [30]:
dataset['month_sin'] = np.sin(2 * np.pi * dataset['month']/12)
dataset['month_cos'] = np.cos(2 * np.pi * dataset['month']/12)
dataset['dow_sin'] = np.sin(2 * np.pi * dataset['dayofweek']/7)
dataset['dow_cos'] = np.cos(2 * np.pi * dataset['dayofweek']/7)

In [ ]:
cols = ['aqi', 'pm2_5', 'pm10', 'no2', 'so2', 'o3', 'temperature_C', 'relative_humidity_%']

In [32]:
lags = [1, 3, 7]
rolls = [3, 7]

In [33]:
dataset = dataset.sort_values(['station', 'date']).reset_index(drop=True)

In [34]:
# create lag feature
for c in cols:
    for lag in lags:
        new_col = f"{c}_lag{lag}"
        dataset[new_col] = dataset.groupby('station')[c].shift(lag)

In [35]:
# crete rolling-mean features
for c in cols:
    for w in rolls:
        new_col = f"{c}_roll{w}"
        dataset[new_col] = (dataset.groupby('station')[c].transform(lambda s: s.rolling(window=w, min_periods=1).mean().shift(1)))

In [36]:
new_features = [c for c in dataset.columns if any(x in c for x in ['_lag', '_roll'])]
print(f"Added {len(new_features)} features: {new_features[:20]}{'' if len(new_features)<=20 else ' ...'}")
print("\nMissing values count for new features:")
print(dataset[new_features].isnull().sum())

Added 40 features: ['aqi_lag1', 'aqi_lag3', 'aqi_lag7', 'pm2_5_lag1', 'pm2_5_lag3', 'pm2_5_lag7', 'pm10_lag1', 'pm10_lag3', 'pm10_lag7', 'no2_lag1', 'no2_lag3', 'no2_lag7', 'so2_lag1', 'so2_lag3', 'so2_lag7', 'o3_lag1', 'o3_lag3', 'o3_lag7', 'temperature_C_lag1', 'temperature_C_lag3'] ...

Missing values count for new features:
aqi_lag1                      15
aqi_lag3                      45
aqi_lag7                     105
pm2_5_lag1                    15
pm2_5_lag3                    45
pm2_5_lag7                   105
pm10_lag1                     15
pm10_lag3                     45
pm10_lag7                    105
no2_lag1                      15
no2_lag3                      45
no2_lag7                     105
so2_lag1                      15
so2_lag3                      45
so2_lag7                     105
o3_lag1                       15
o3_lag3                       45
o3_lag7                      105
temperature_C_lag1            15
temperature_C_lag3            45
temperatur

In [37]:
initial_len = len(dataset)
dataset = dataset[~dataset[new_features].isnull().any(axis=1)].reset_index(drop=True)
dropped = initial_len - len(dataset)
print(f"Dropped {dropped} rows ({dropped/initial_len*100:.2f}%) because lag/roll features were not available.")

Dropped 105 rows (1.07%) because lag/roll features were not available.


In [38]:
dataset['station'].nunique()

15

In [39]:
dataset = pd.get_dummies(dataset, columns=['station'], prefix='st', drop_first=False)

In [40]:
train_end = '2025-05-30'
valid_end = '2025-08-30'

In [41]:
train_data = dataset[ dataset['date'] <= train_end ]
valid_data = dataset[ (dataset['date'] > train_end) & (dataset['date'] <= valid_end) ]
test_data = dataset[ dataset['date'] > valid_end ]

In [42]:
target = 'aqi'
features = [c for c in dataset.columns if c not in ['date', 'aqi']]

In [43]:
x_train = train_data[features]
y_train = train_data[target]

x_valid = valid_data[features]
y_valid = valid_data[target]

x_test = test_data[features]
y_test = test_data[target]

In [44]:
from sklearn.ensemble import RandomForestRegressor

In [45]:
model = RandomForestRegressor(n_estimators=500, criterion='squared_error', max_depth=20)
model.fit(x_train, y_train)

,n_estimators,500
,criterion,'squared_error'
,max_depth,20
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [46]:
y_train_pred = model.predict(x_train)
y_valid_pred = model.predict(x_valid)
y_test_pred = model.predict(x_test)

In [47]:
from sklearn.metrics import r2_score

In [48]:
train_score = r2_score(y_train, y_train_pred)
valid_score = r2_score(y_valid, y_valid_pred)
test_score = r2_score(y_test, y_test_pred)

In [49]:
print(f"\nModel Performance:")
print(f"Train R² Score: {train_score:.4f}")
print(f"Validation R² Score: {valid_score:.4f}")
print(f"Test R² Score: {test_score:.4f}")


Model Performance:
Train R² Score: 0.9720
Validation R² Score: 0.6138
Test R² Score: 0.7213


In [50]:
# Calculate feature importance
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nTop 10 Most Important Features:")
print(feature_importance.head(10))


Top 10 Most Important Features:
                      feature  importance
2                       pm2_5    0.841314
4                         no2    0.008157
3                        pm10    0.007351
26                   no2_lag1    0.006959
30                   so2_lag3    0.006638
10                        day    0.005589
55  relative_humidity_%_roll3    0.005254
28                   no2_lag7    0.005004
32                    o3_lag1    0.004969
8         relative_humidity_%    0.004807
